---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

# 🔁 Cross-Validation & Resampling
**ISLP Chapter 5 · Pattern Recognition for the Rest of Us**

> Jump in — each section builds on the last. Run cells top to bottom with Shift+Enter.

### What you'll learn
- Why a single train/test split is not enough
- k-Fold Cross-Validation
- Leave-One-Out Cross-Validation (LOOCV)
- Bootstrap for uncertainty estimation

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first before any others
# (Tip: Runtime → Run all  OR  Shift+Enter from the top)
import pandas as pd
auto = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Auto.csv').dropna()
print(f"Auto: {auto.shape}  |  Columns: {list(auto.columns)}")
auto.head()

## 📐 Part 1 — The Validation Set Approach

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

## 🔬 Part 2 — k-Fold Cross-Validation

In [ ]:
# Visualize how K-fold CV works
fig, ax = plt.subplots(figsize=(11, 4))
n_obs, K = 30, 5
fold_size = n_obs // K

colors = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1','#0e7490']

for k in range(K):
    for i in range(n_obs):
        fold = i // fold_size
        if fold == k:
            color = colors[k]
            label = 'Validation' if i == k * fold_size else ''
        else:
            color = '#d0d8e8'
            label = ''
        ax.barh(k, 1, left=i, color=color, edgecolor='white', height=0.7)

ax.set_yticks(range(K))
ax.set_yticklabels([f'Fold {k+1}' for k in range(K)])
ax.set_xlabel('Observation index')
ax.set_title('5-Fold Cross-Validation — colored = validation fold, gray = training')

# Legend patches
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=f'Fold {i+1} as validation') for i in range(K)]
legend_elements.append(Patch(facecolor='#d0d8e8', label='Training'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 📊 Part 3 — LOOCV vs k-Fold Comparison

In [ ]:
# LOOCV vs 5-fold vs 10-fold — compare estimates and runtime
import time

results = {}
for name, cv in [('LOOCV', LeaveOneOut()),
                 ('5-Fold CV', KFold(5, shuffle=True, random_state=42)),
                 ('10-Fold CV', KFold(10, shuffle=True, random_state=42))]:
    pipe = Pipeline([('poly', PolynomialFeatures(2)), ('lr', LinearRegression())])
    t0 = time.time()
    scores = cross_val_score(pipe, X_auto, y_auto, cv=cv,
                             scoring='neg_mean_squared_error')
    elapsed = time.time() - t0
    results[name] = {'mse': -scores.mean(), 'std': scores.std(), 'time': elapsed}

print("=" * 55)
print(f"{'Method':<15} {'CV MSE':>10} {'Std':>10} {'Time (s)':>10}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<15} {r['mse']:>10.3f} {r['std']:>10.3f} {r['time']:>10.3f}")
print("=" * 55)
print("\n📌 All three give similar MSE estimates.")
print("   LOOCV is slower and has higher std — 10-fold is the standard choice.")

## ✅ Part 4 — Bootstrap

In [ ]:
# Bootstrap example: estimate standard error of the median
np.random.seed(42)
# Simulate a skewed distribution (e.g., income)
population_sample = np.concatenate([
    np.random.exponential(scale=50000, size=200),
    np.random.normal(200000, 30000, 20)   # a few high earners
])
population_sample = population_sample[population_sample > 0]
n = len(population_sample)

# Bootstrap
B = 2000
bootstrap_medians = []
for _ in range(B):
    boot_sample = np.random.choice(population_sample, size=n, replace=True)
    bootstrap_medians.append(np.median(boot_sample))

se_bootstrap = np.std(bootstrap_medians)
observed_median = np.median(population_sample)
ci_lower, ci_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: data distribution
axes[0].hist(population_sample, bins=40, color='#1e5fa8', alpha=0.7, edgecolor='white')
axes[0].axvline(observed_median, color='#e85d20', lw=2, label=f'Median = ${observed_median:,.0f}')
axes[0].set_title('Income Distribution (n=220)')
axes[0].set_xlabel('Income ($)')
axes[0].legend()

# Right: bootstrap distribution of median
axes[1].hist(bootstrap_medians, bins=50, color='#1a7a45', alpha=0.7, edgecolor='white')
axes[1].axvline(observed_median, color='#e85d20', lw=2, label=f'Observed median')
axes[1].axvline(ci_lower, color='#888', lw=1.5, ls='--')
axes[1].axvline(ci_upper, color='#888', lw=1.5, ls='--', label=f'95% CI: [${ci_lower:,.0f}, ${ci_upper:,.0f}]')
axes[1].set_title(f'Bootstrap Distribution of Median (B={B})')
axes[1].set_xlabel('Bootstrap Median ($)')
axes[1].legend(fontsize=9)

plt.suptitle('Bootstrap — Estimating Uncertainty in the Median', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nObserved median:    ${observed_median:,.0f}")
print(f"Bootstrap SE:       ${se_bootstrap:,.0f}")
print(f"95% CI:             [${ci_lower:,.0f}, ${ci_upper:,.0f}]")
print("\n📌 We never collected new data — the bootstrap created 2000 synthetic datasets from the one we have.")

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

# Load datasets
ads = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Advertising.csv')
auto = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Auto.csv', na_values='?').dropna()
print(f'✓ Advertising: {ads.shape}  columns: {ads.columns.tolist()}')
print(f'✓ Auto: {auto.shape}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 💼 Exercise

Using the Auto dataset, use 10-fold CV to select the optimal polynomial degree (1–8) for predicting `mpg` from `horsepower`. Compare CV error to test error. How close is the CV estimate?

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — Cross-Validation & Resampling
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does K-fold CV do that a single train/test split does not?
# @markdown **Q2:** Why is LOOCV sometimes worse than 10-fold CV despite being less biased?
# @markdown **Q3:** In bootstrap resampling, approximately what fraction of observations are never sampled?
# @markdown **Q4:** You have 20 observations. Which CV method would you choose and why?
# @markdown **Q5:** How does CV help with model selection? Give a specific example.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Cross-Validation & Resampling"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*